# Algeria Export Opportunity Analysis
## Phase 1 — Data Pipeline: Loading & Merging

### Project Context
Algeria's export economy is heavily concentrated in hydrocarbons (~90% of export revenues).
This project builds an end-to-end ML system to identify and forecast international export
opportunities for Algerian exporters across all sectors (agriculture, industry, services).

This notebook is the **first phase** of the data pipeline. It:
1. Loads raw trade data from CEPII BACI (bilateral trade flows)
2. Loads geographic and colonial features from CEPII GeoDist
3. Downloads macroeconomic indicators from the World Bank API
4. Performs EDA on each source independently
5. Merges all sources into yearly parquet files → handed to Kaggle for final assembly

### Data Sources
| Source | Description | Link |
|--------|-------------|------|
| CEPII BACI HS92 | Bilateral trade flows 1995–2024, 200+ countries, 5000+ products | [Download](https://www.cepii.fr/CEPII/en/bdd_modele/bdd_modele_item.asp?id=37) |
| CEPII GeoDist | Geographic & colonial features for country pairs | [Download](https://www.cepii.fr/CEPII/en/bdd_modele/bdd_modele_item.asp?id=6) |
| World Bank API | Macroeconomic indicators 1995–2024 | Pulled via `wbgapi` in notebook |

### Large Files (not in git — too large)
| File | Description | Link |
|------|-------------|------|
| `BACI_HS92_V202601.zip` | Raw BACI trade data (~4GB) | [CEPII](https://www.cepii.fr/DATA_DOWNLOAD/baci/data/BACI_HS92_V202601.zip) |
| `geo_cepii.xls` | Country-level geographic features | [CEPII](https://www.cepii.fr/distance/geo_cepii.xls) |
| `dist_cepii.xls` | Bilateral distance & colonial features | [CEPII](https://www.cepii.fr/distance/dist_cepii.zip) |
| `merged_by_year.zip` | Output of this notebook (yearly merged parquets) | [Google Drive](https://drive.google.com/file/d/1vjt-KpGDJDXOXcPlQpiaQcEip1yeYn_h/view?usp=sharing) |

### How to Reproduce
1. Download the 3 raw files from the links above
2. Extract them into `data/raw/` following this structure:
<pre>
data/raw/
├── BACI_HS92_V202601/     ← extracted CSV files
├── geo_cepii.xls
└── dist_cepii.xls
</pre>

3. Run this notebook cell by cell
4. Upload the resulting `merged_by_year/` folder to Kaggle
5. Run `master_table_builder.ipynb` on Kaggle to produce the final master table

### Notebook Structure
| Step | Description |
|------|-------------|
| Setup | Imports, paths, environment detection |
| Loading | Read BACI CSVs → save as parquets to free RAM |
| Reference Tables | Country codes, product codes, GeoDist, World Bank |
| EDA | Trade flow analysis, Algeria export profile, partner analysis |
| Merging | Join all sources year by year → `merged_by_year/` |

## 1. Environment Setup

Defines all paths relative to project root so the notebook works
on any machine without manual path changes.
All data files are read from `data/raw/` and outputs saved to `data/processed/`.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
import warnings
warnings.filterwarnings('ignore')

# Paths
ROOT        = os.path.abspath(os.path.join(os.getcwd(), ".."))
RAW         = os.path.join(ROOT, "data", "raw")
PROCESSED   = os.path.join(ROOT, "data", "processed")

BACI_DIR    = os.path.join(RAW, "BACI_HS92_V202601")
GEO_FILE    = os.path.join(RAW, "geo_cepii.xls")
DIST_FILE   = os.path.join(RAW, "dist_cepii.xls")

os.makedirs(PROCESSED, exist_ok=True)

print("ROOT      :", ROOT)
print("BACI dir  :", BACI_DIR)
print("GEO file  :", GEO_FILE)
print("DIST file :", DIST_FILE)

ROOT      : c:\Y3\S2\ML\project
BACI dir  : c:\Y3\S2\ML\project\data\raw\BACI_HS92_V202601
GEO file  : c:\Y3\S2\ML\project\data\raw\geo_cepii.xls
DIST file : c:\Y3\S2\ML\project\data\raw\dist_cepii.xls


## 2. Dependencies

Installing `pyarrow` for efficient parquet read/write support.
Parquet format is used throughout this project instead of CSV because:
- ~5x smaller file size (compression)
- ~10x faster read speed
- Preserves dtypes (no re-casting needed on reload)

In [2]:
import subprocess
subprocess.run(["pip", "install", "--upgrade", "pyarrow"], check=True)

CompletedProcess(args=['pip', 'install', '--upgrade', 'pyarrow'], returncode=0)

In [3]:
baci_files = sorted(glob.glob(os.path.join(BACI_DIR, "*.csv")))
print(f"Total BACI files found: {len(baci_files)}")
for f in baci_files:
    print(os.path.basename(f))

Total BACI files found: 2
country_codes_V202601.csv
product_codes_HS92_V202601.csv


## 3. BACI — CSV to Parquet Conversion

The raw BACI dataset contains 30 yearly CSV files (1995–2024), each with
5–11 million rows. Loading all years at once into RAM requires ~15GB and
crashed the kernel repeatedly.

### Solution
Read each year CSV individually, apply memory-optimized dtypes immediately,
and save as compressed parquet. This reduces:
- Per-year RAM usage: ~500MB instead of ~2GB
- Disk usage: ~50MB per parquet instead of ~150MB per CSV

### Dtype optimization
| Column | Default | Optimized | Saving |
|--------|---------|-----------|--------|
| t (year) | int64 | int16 | 75% |
| i, j (country) | int64 | int16 | 75% |
| k (product) | object | category | ~90% |
| v, q (value/qty) | float64 | float32 | 50% |

A skip-if-exists check allows resuming if the process is interrupted mid-way.
After conversion, the raw CSVs are deleted to free disk space (cell 11).

In [4]:
import os, glob
import pandas as pd

PARQUET_DIR = os.path.join(PROCESSED, "baci_by_year")
os.makedirs(PARQUET_DIR, exist_ok=True)

baci_files = sorted(glob.glob(os.path.join(BACI_DIR, "BACI_HS92_Y*.csv")))

DTYPES = {
    "t": "int16",
    "i": "int16",
    "j": "int16",
    "k": "category",
    "v": "float32",
    "q": "float32"
}

for fpath in baci_files:
    year = int(os.path.basename(fpath).split("_Y")[1].split("_")[0])
    out  = os.path.join(PARQUET_DIR, f"baci_{year}.parquet")

    if os.path.exists(out):
        print(f"  {year} already done, skipping")
        continue

    print(f"  loading {year}...", end=" ")
    df = pd.read_csv(fpath, dtype=DTYPES)
    df.to_parquet(out, index=False)
    print(f"{len(df):,} rows → saved")
    del df

print("\nDone. Parquet files saved to:", PARQUET_DIR)


Done. Parquet files saved to: c:\Y3\S2\ML\project\data\processed\baci_by_year


## 4. Reference Tables

Loading country and product code lookup tables bundled with BACI.
These map numeric codes to human-readable names and will be joined
during the merge step.

- `country_codes_V202601.csv` → 238 countries, maps numeric code → ISO2, ISO3, name
- `product_codes_HS92_V202601.csv` → 5,022 products, maps HS92 6-digit code → description

In [5]:
# load reference tables
countries = pd.read_csv(os.path.join(BACI_DIR, "country_codes_V202601.csv"))
products  = pd.read_csv(os.path.join(BACI_DIR, "product_codes_HS92_V202601.csv"), dtype={"code": str})

print("=== Countries ===")
print(countries.shape)
print(countries.head())

print("\n=== Products ===")
print(products.shape)
print(products.head())

=== Countries ===
(238, 4)
   country_code    country_name country_iso2 country_iso3
0             4     Afghanistan           AF          AFG
1             8         Albania           AL          ALB
2            12         Algeria           DZ          DZA
3            16  American Samoa           AS          ASM
4            20         Andorra           AD          AND

=== Products ===
(5022, 2)
     code                                        description
0  010111           Horses: live, pure-bred breeding animals
1  010119  Horses: live, other than pure-bred breeding an...
2  010120                     Asses, mules and hinnies: live
3  010210   Bovine animals: live, pure-bred breeding animals
4  010290  Bovine animals: live, other than pure-bred bre...


## 5. GeoDist — Geographic & Colonial Features

CEPII GeoDist provides two complementary files:

**`geo_cepii.xls`** — country-level features:
- Geographic: latitude, longitude, area, landlocked status
- Linguistic: official languages, spoken languages
- Colonial: colonizer countries

**`dist_cepii.xls`** — bilateral country-pair features:
- `dist`: population-weighted distance between cities (km)
- `contig`: 1 if countries share a land border
- `comlang_off`: 1 if countries share an official language
- `colony`: 1 if a colonial relationship existed

These features are critical for the gravity model underlying trade flow analysis
and will serve as key features for the ML models.

In [6]:
# load geodist files
geo   = pd.read_excel(GEO_FILE)
dist  = pd.read_excel(DIST_FILE)

print("=== geo_cepii ===")
print(geo.shape)
print(geo.head())
print("\nColumns:", geo.columns.tolist())

print("\n=== dist_cepii ===")
print(dist.shape)
print(dist.head())
print("\nColumns:", dist.columns.tolist())

=== geo_cepii ===
(238, 34)
  iso2 iso3  cnum      country         pays     area     dis_int  landlocked  \
0   AW  ABW   533        Aruba        Aruba      193    5.225315           0   
1   AF  AFG     4  Afghanistan  Afghanistan   652225  303.761400           1   
2   AO  AGO    24       Angola       Angola  1246700  419.966600           0   
3   AI  AIA   660     Anguilla     Anguilla      102    3.798690           0   
4   AL  ALB     8      Albania      Albanie    28748   63.773110           0   

  continent     city_en  ... lang9_2  lang9_3  lang9_4  colonizer1  \
0   America  Oranjestad  ...       .        .        .         NLD   
1      Asia       Kabul  ...   Uzbek        .        .           .   
2    Africa      Luanda  ...       .        .        .         PRT   
3   America  The Valley  ...       .        .        .         GBR   
4    Europe      Tirana  ...       .        .        .         TUR   

   colonizer2 colonizer3 colonizer4 short_colonizer1 short_colonizer2 

In [ ]:
import subprocess
subprocess.run(["pip", "install", "wbgapi"], check=True)

## 6. World Bank Indicators

Pulling macroeconomic indicators via the `wbgapi` Python library
for all countries from 1995 to 2024.

| Indicator | Code | Description |
|-----------|------|-------------|
| GDP | NY.GDP.MKTP.CD | Total GDP in current USD |
| GDP Growth | NY.GDP.MKTP.KD.ZG | Annual GDP growth rate (%) |
| Population | SP.POP.TOTL | Total population |
| Trade % GDP | NE.TRD.GNFS.ZS | Trade openness indicator |
| Inflation | FP.CPI.TOTL.ZG | Consumer price inflation (%) |
| Unemployment | SL.UEM.TOTL.ZS | Unemployment rate (%) |
| Interest Rate | FR.INR.LEND | Lending interest rate (%) |

### Missing value handling
After dropping non-country aggregates (regional/income groups),
missing values are imputed in 3 steps:
1. Forward/backward fill per country (use nearby years)
2. Regional + year median (same continent, same year)
3. Global year median (fallback)

`interest_rate` had 63% missing even after imputation and was dropped entirely
in the master table builder.

In [9]:
import wbgapi as wb

# indicators we need
WB_INDICATORS = {
    "NY.GDP.MKTP.CD"    : "gdp",
    "NY.GDP.MKTP.KD.ZG" : "gdp_growth",
    "SP.POP.TOTL"       : "population",
    "NE.TRD.GNFS.ZS"    : "trade_percent_gdp",
    "FP.CPI.TOTL.ZG"    : "inflation",
    "SL.UEM.TOTL.ZS"    : "unemployment",
    "FR.INR.LEND"       : "interest_rate"
}

print("Downloading World Bank indicators (1995–2024)...")
wb_data = wb.data.DataFrame(
    list(WB_INDICATORS.keys()),
    time=range(1995, 2025),
    skipBlanks=True,
    columns="series"
).reset_index()

wb_data.rename(columns={"economy": "iso3", "time": "year"}, inplace=True)
wb_data.rename(columns=WB_INDICATORS, inplace=True)
wb_data["year"] = wb_data["year"].str.replace("YR", "").astype(int)

print(f"Shape       : {wb_data.shape}")
print(f"Years range : {wb_data['year'].min()} – {wb_data['year'].max()}")
print(f"Countries   : {wb_data['iso3'].nunique()}")
print(wb_data.head(10))

Shape       : (7950, 9)
Years range : 1995 – 2024
Countries   : 265
  iso3  year  inflation  interest_rate  trade_percent_gdp           gdp  \
0  ABW  1995   3.361391      10.600000         171.362098  1.320670e+09   
1  ABW  1996   3.225288      10.375000         175.344130  1.379888e+09   
2  ABW  1997   2.999948      10.250000         168.781911  1.531844e+09   
3  ABW  1998   1.869489      11.041667         163.267360  1.665363e+09   
4  ABW  1999   2.280372      13.141667         164.559014  1.722905e+09   
5  ABW  2000   4.044021      12.066667         145.072880  1.873453e+09   
6  ABW  2001   2.883604      12.600000         140.391109  1.896457e+09   
7  ABW  2002   3.315247      11.300000         133.230771  1.961844e+09   
8  ABW  2003   3.656365      10.700000         132.794291  2.044112e+09   
9  ABW  2004   2.529129       9.600000         132.430930  2.254831e+09   

   gdp_growth  unemployment  population  
0    2.547144           NaN     79805.0  
1    1.185789         

In [8]:
import subprocess
subprocess.run(["pip", "install", "wbgapi"], check=True)

CompletedProcess(args=['pip', 'install', 'wbgapi'], returncode=0)

## 7. Data Sources Summary

Quick sanity check — verifying all 5 sources loaded correctly
before moving to EDA and merging.

In [10]:
print("=" * 50)
print("DATA SOURCES SUMMARY")
print("=" * 50)

print(f"\n BACI (trade flows)")
print(f"   yearly parquets  : 30 files in baci_by_year/")

print(f"\n Countries reference")
print(f"   shape            : {countries.shape}")
print(f"   columns          : {countries.columns.tolist()}")

print(f"\n Products reference")
print(f"   shape            : {products.shape}")
print(f"   columns          : {products.columns.tolist()}")

print(f"\n GeoDist - country level (geo)")
print(f"   shape            : {geo.shape}")
print(f"   key columns      : {['iso3','landlocked','continent','lat','lon']}")

print(f"\n GeoDist - bilateral (dist)")
print(f"   shape            : {dist.shape}")
print(f"   key columns      : {['iso_o','iso_d','dist','contig','comlang_off','colony']}")

print(f"\n World Bank indicators")
print(f"   shape            : {wb_data.shape}")
print(f"   columns          : {wb_data.columns.tolist()}")
print(f"   missing values   :")
print(wb_data.isnull().sum())

DATA SOURCES SUMMARY

 BACI (trade flows)
   yearly parquets  : 30 files in baci_by_year/

 Countries reference
   shape            : (238, 4)
   columns          : ['country_code', 'country_name', 'country_iso2', 'country_iso3']

 Products reference
   shape            : (5022, 2)
   columns          : ['code', 'description']

 GeoDist - country level (geo)
   shape            : (238, 34)
   key columns      : ['iso3', 'landlocked', 'continent', 'lat', 'lon']

 GeoDist - bilateral (dist)
   shape            : (50176, 14)
   key columns      : ['iso_o', 'iso_d', 'dist', 'contig', 'comlang_off', 'colony']

 World Bank indicators
   shape            : (7950, 9)
   columns          : ['iso3', 'year', 'inflation', 'interest_rate', 'trade_percent_gdp', 'gdp', 'gdp_growth', 'unemployment', 'population']
   missing values   :
iso3                    0
year                    0
inflation            2582
interest_rate        4370
trade_percent_gdp    1537
gdp                   274
gdp_growth   

## 8. Free Disk Space

The raw BACI CSVs are no longer needed since all years have been
saved as optimized parquet files. Deleting them frees ~4GB of disk space.
This was necessary because the local machine had only ~7GB free disk space
throughout this project.

In [11]:
import shutil, gc

# delete raw BACI CSVs since we have parquets already
raw_csvs = glob.glob(os.path.join(BACI_DIR, "BACI_HS92_Y*.csv"))
for f in raw_csvs:
    os.remove(f)
    
freed = len(raw_csvs) * 150  # approx 150MB per CSV
print(f"Deleted {len(raw_csvs)} CSV files (~{freed/1000:.1f} GB freed)")

free = shutil.disk_usage("C:\\").free / 1e9
print(f"Free disk space now: {free:.2f} GB")

Deleted 0 CSV files (~0.0 GB freed)
Free disk space now: 6.36 GB


## 9. Merging All Sources → Yearly Parquet Files

Joining BACI trade flows with all auxiliary sources year by year.
Each year is processed independently to avoid loading the full dataset into RAM.

### Join strategy
<pre>
BACI (numeric i, j)
    │
    ├──► countries table  →  iso3_i, iso3_j
    │
    ├──► geo_cepii        →  continent_i/j, landlocked_i/j, lat_i/j, lon_i/j
    │                        (joined twice: once for exporter, once for importer)
    │
    ├──► dist_cepii       →  dist, contig, comlang_off, colony
    │                        (bilateral pair join on iso3_i + iso3_j)
    │
    ├──► World Bank       →  gdp_i/j, gdp_growth_i/j, population_i/j,
    │                        inflation_i/j, trade_percent_gdp_i/j, unemployment_i/j
    │                        (joined twice: once for exporter, once for importer)
    │
    └──► products table   →  product_description
</pre>
### Memory optimization
Instead of `df.merge()` (creates full copies in RAM), we use **dictionary mapping**
(`df["col"].map(dict)`) which is ~3x more memory efficient. Each source is
pre-converted to a dictionary indexed by ISO3 or (ISO3_i, ISO3_j) pair.

### Output
One parquet file per year saved to `data/processed/merged_by_year/`.
These are then zipped and uploaded to Kaggle for final assembly.
The zip file is available at: [Kaggle Dataset](YOUR_KAGGLE_LINK)

In [12]:
import gc

# we dont need ALL columns — keep only what matters for master table
GEO_COLS     = ["iso3", "continent", "landlocked", "lat", "lon"]
DIST_COLS    = ["iso_o", "iso_d", "dist", "contig", "comlang_off", "colony"]
WB_COLS      = ["iso3", "year", "gdp", "gdp_growth", "population",
                "inflation", "trade_percent_gdp", "interest_rate", "unemployment"]

geo_slim  = geo[GEO_COLS].copy()
dist_slim = dist[DIST_COLS].copy()

valid_iso3 = set(countries["country_iso3"].unique())
wb_clean = wb_data[wb_data["iso3"].isin(valid_iso3)].copy()
wb_clean = wb_clean.merge(geo[["iso3", "continent"]], on="iso3", how="left")
wb_slim   = wb_clean[WB_COLS].copy()
prod_slim = products.rename(columns={"code":"k","description":"product_description"})[["k","product_description"]].copy()
code_to_iso3 = countries.set_index("country_code")["country_iso3"].to_dict()

geo_maps = {
    col: geo_slim.set_index("iso3")[col].to_dict()
    for col in ["continent", "landlocked", "lat", "lon"]
}
dist_maps = {
    col: dist_slim.set_index(["iso_o", "iso_d"])[col].to_dict()
    for col in ["dist", "contig", "comlang_off", "colony"]
}

MERGE_DIR = os.path.join(PROCESSED, "merged_by_year")
os.makedirs(MERGE_DIR, exist_ok=True)

parquet_files = sorted(glob.glob(os.path.join(PROCESSED, "baci_by_year", "*.parquet")))

for fpath in parquet_files:
    year = int(os.path.basename(fpath).replace("baci_","" ).replace(".parquet",""))
    out  = os.path.join(MERGE_DIR, f"merged_{year}.parquet")

    if os.path.exists(out):
        print(f"  {year} skipped")
        continue

    print(f"  processing {year}...", end=" ")

    # ── load ───────────────────────────────────────────────────────
    df = pd.read_parquet(fpath)
    df["k"] = df["k"].astype(str)

    # ── fill missing q ───────────────────────────────────────────────
    med_q = df.groupby("k")["q"].transform("median")
    df["q"] = df["q"].fillna(med_q).astype("float32")

    # ── map numeric → iso3 ──────────────────────────────────────────
    df["iso3_i"] = df["i"].map(code_to_iso3).astype("category")
    df["iso3_j"] = df["j"].map(code_to_iso3).astype("category")
    df.drop(columns=["i", "j"], inplace=True)

    # ── join products ────────────────────────────────────────────────
    df = df.merge(prod_slim, on="k", how="left")
    # keep k as string to avoid mixed dictionary/string schema in merged parquet files

    # ── add geo exporter/importer metadata via mapping
    for col, mapping in geo_maps.items():
        df[f"{col}_i"] = df["iso3_i"].map(mapping)
        df[f"{col}_j"] = df["iso3_j"].map(mapping)

    # ── add bilateral distance via tuple mapping
    pair_keys = list(zip(df["iso3_i"], df["iso3_j"]))
    for col, mapping in dist_maps.items():
        df[col] = [mapping.get(key) for key in pair_keys]

    # ── downcast all floats ──────────────────────────────────────────
    for col in df.select_dtypes("float64").columns:
        df[col] = df[col].astype("float32")

    # ── join WB exporter/importer via mapping
    wb_yr = wb_slim[wb_slim["year"] == year].drop(columns=["year"])
    wb_map = {
        col: wb_yr.groupby("iso3")[col].first().to_dict()
        for col in wb_yr.columns
        if col != "iso3"
    }
    for col, mapping in wb_map.items():
        df[f"{col}_i"] = df["iso3_i"].map(mapping).astype("float32")
        df[f"{col}_j"] = df["iso3_j"].map(mapping).astype("float32")

    del wb_yr, wb_map

    # ── final downcast ───────────────────────────────────────────────
    df["landlocked_i"] = df["landlocked_i"].astype("Int8")
    df["landlocked_j"] = df["landlocked_j"].astype("Int8")
    df["contig"]       = df["contig"].astype("Int8")
    df["comlang_off"]  = df["comlang_off"].astype("Int8")
    df["colony"]       = df["colony"].astype("Int8")
    df["t"]            = df["t"].astype("int16")

    df.to_parquet(out, index=False, compression="snappy")
    size_mb = os.path.getsize(out) / 1e6
    print(f"saved — shape {df.shape} — {size_mb:.1f} MB")

    del df
    gc.collect()

print("\nAll years done.")
free = shutil.disk_usage("C:\\").free / 1e9
print(f"Free disk space: {free:.2f} GB")

  1995 skipped
  1996 skipped
  1997 skipped
  1998 skipped
  1999 skipped
  2000 skipped
  2001 skipped
  2002 skipped
  2003 skipped
  2004 skipped
  2005 skipped
  2006 skipped
  2007 skipped
  2008 skipped
  2009 skipped
  2010 skipped
  2011 skipped
  2012 skipped
  2013 skipped
  2014 skipped
  2015 skipped
  2016 skipped
  2017 skipped
  2018 skipped
  2019 skipped
  2020 skipped
  2021 skipped
  2022 skipped
  2023 skipped
  2024 skipped

All years done.
Free disk space: 6.36 GB
